# Movie Dataset Data Cleaning & EDA Project

This notebook is designed for a complete data preprocessing and exploratory data analysis (EDA) pipeline. We will inspect the dataset, handle missing values and duplicates, treat outliers, and perform univariate and multivariate analysis.

## 1. Import Libraries & Setup

Import necessary libraries for data manipulation and visualization.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

## 2. Data Loading & Initial Inspection

Load the dataset and understand its shape, columns, and basic descriptive statistics.

In [ ]:
df = pd.read_csv('movies_modified.csv')
df.head()

In [ ]:
print("Dataset Shape:", df.shape)

In [ ]:
df.columns

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.sample(5)

## 3. Data Cleaning: Handling Duplicates

Identify and remove duplicate records to ensure data integrity.

In [ ]:
print("Total duplicated rows:", df.duplicated().sum())

In [ ]:
duplicate_rows = df[df.duplicated()]
duplicate_rows.head()

In [ ]:
df = df.drop_duplicates()

In [ ]:
print("Total duplicated rows after dropping:", df.duplicated().sum())

## 4. Data Cleaning: Missing Value Imputation

Identify missing values visually and computationally. We will impute numerical missing values using the median and categorical ones using the mode.

In [ ]:
df.isnull().sum()

In [ ]:
plt.figure(figsize=(10,5))
sns.heatmap(df.isnull(), cbar=False, cmap='viridis')
plt.title('Missing Values Heatmap')
plt.show()

In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns
categorical_cols = df.select_dtypes(exclude=np.number).columns
print("Numeric Columns:", numeric_cols.tolist())
print("Categorical Columns:", categorical_cols.tolist())

In [ ]:
# Impute numeric columns with median
for col in numeric_cols:
    df[col].fillna(df[col].median(), inplace=True)

# Impute categorical columns with mode
for col in categorical_cols:
    df[col].fillna(df[col].mode()[0], inplace=True)

In [ ]:
print("Missing values after imputation:\n", df.isnull().sum())

## 5. Data Cleaning: Outlier Treatment

Identify outliers in key continuous variables using boxplots and cap them using the Interquartile Range (IQR) method.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.boxplot(x=df['budget'], ax=axes[0]).set_title('Budget Before Outlier Treatment')
sns.boxplot(x=df['gross'], ax=axes[1]).set_title('Gross Before Outlier Treatment')
plt.show()

In [ ]:
def cap_outliers(dataframe, column):
    Q1 = dataframe[column].quantile(0.25)
    Q3 = dataframe[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    # Capping outliers
    dataframe[column] = np.where(dataframe[column] < lower_bound, lower_bound, dataframe[column])
    dataframe[column] = np.where(dataframe[column] > upper_bound, upper_bound, dataframe[column])
    return dataframe

# Apply IQR Capping to budget and gross
df = cap_outliers(df, 'budget')
df = cap_outliers(df, 'gross')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.boxplot(x=df['budget'], ax=axes[0]).set_title('Budget After Outlier Treatment')
sns.boxplot(x=df['gross'], ax=axes[1]).set_title('Gross After Outlier Treatment')
plt.show()

## 6. Exploratory Data Analysis (EDA): Univariate Analysis

Analyze the distribution of individual variables using histograms and count plots.

In [ ]:
df['budget'].hist(bins=30, figsize=(8,4))
plt.title('Budget Distribution')
plt.show()

In [ ]:
df['gross'].hist(bins=30, figsize=(8,4))
plt.title('Gross Earnings Distribution')
plt.show()

In [ ]:
df['score'].hist(bins=30, figsize=(8,4))
plt.title('Movie Score Distribution')
plt.show()

In [ ]:
plt.figure(figsize=(10,5))
sns.countplot(y=df['rating'], order=df['rating'].value_counts().index)
plt.title('Movie Count by Rating')
plt.show()

In [ ]:
plt.figure(figsize=(10,6))
df['genre'].value_counts().head(10).plot(kind='bar')
plt.title('Top 10 Genres')
plt.ylabel('Count')
plt.show()

In [ ]:
plt.figure(figsize=(10,6))
df['company'].value_counts().head(10).plot(kind='bar')
plt.title('Top 10 Production Companies')
plt.ylabel('Count')
plt.show()

## 7. Exploratory Data Analysis (EDA): Bivariate & Multivariate Analysis

Understand the relationships between different variables using scatter plots, correlations, and aggregation techniques.

In [ ]:
plt.figure(figsize=(10,6))
plt.scatter(df['budget'], df['gross'], alpha=0.5)
plt.title('Budget vs Gross Earnings')
plt.xlabel('Budget')
plt.ylabel('Gross')
plt.show()

In [ ]:
corr = df.select_dtypes(include=np.number).corr()
plt.figure(figsize=(8,6))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Numerical Features Correlation Matrix')
plt.show()

In [ ]:
sns.pairplot(df[['budget','gross','score','runtime']])
plt.show()

### Encoding Categorical Features for Full Correlation Matrix

In [ ]:
from sklearn.preprocessing import LabelEncoder
df_numerized = df.copy()
le = LabelEncoder()

for col in df_numerized.columns:
    if df_numerized[col].dtype == 'object':
        df_numerized[col] = le.fit_transform(df_numerized[col].astype(str))

plt.figure(figsize=(14,10))
sns.heatmap(df_numerized.corr(), annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Full Correlation Matrix (Including Encoded Categoricals)')
plt.show()

### Aggregation & Trend Analysis

In [ ]:
plt.figure(figsize=(10,5))
df.groupby('year').size().plot()
plt.title('Number of Movies Released Per Year')
plt.ylabel('Count')
plt.show()

In [ ]:
plt.figure(figsize=(10,6))
df.groupby('genre')['gross'].mean().sort_values(ascending=False).head(10).plot(kind='bar')
plt.title('Top 10 Genres by Average Gross')
plt.ylabel('Average Gross')
plt.show()

In [ ]:
plt.figure(figsize=(10,6))
df.groupby('rating')['score'].mean().sort_values().plot(kind='barh')
plt.title('Average Score by Age Rating')
plt.xlabel('Average Score')
plt.show()

In [ ]:
print("Top 10 Movies by Gross Earnings:")
top_movies = df.sort_values('gross', ascending=False).head(10)
top_movies[['name', 'gross', 'year', 'director']]

## 8. Export Cleaned Dataset

Save the cleaned and processed dataset for modeling or presentation.

In [ ]:
df.to_csv('movies_cleaned.csv', index=False)
print("Cleaned dataset saved as 'movies_cleaned.csv'.")
df.head()